## {RQ2 - Prediction and Explanation:} 
Can we predict spatial variation in racialised discourse from structural factors (pool attributes, 
constituency demographics, political orientation)? What does model performance reveal about the 
method's explanatory power and potential for identifying intervention sites?

In [ ]:
# Phase 0: Setup
# ==============

import pandas as pd
import numpy as np
import os
import pyarrow.parquet as pq
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
from scipy import stats
from scipy.spatial.distance import pdist, squareform
import geopandas as gpd
from shapely.geometry import Point, shape
import json

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Statistics & Modeling
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (roc_auc_score, mean_squared_error, r2_score,
                             accuracy_score, confusion_matrix, classification_report)
from sklearn.inspection import permutation_importance
import xgboost as xgb

# Spatial analysis
from pysal.lib import weights
from pysal.explore import esda
from splot.esda import plot_moran, lisa_cluster

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print("Libraries loaded successfully!")

## Phase 1.1: Load Data

In [ ]:
def read_parquet_with_checks(file_path):
    print(f"File size: {os.path.getsize(file_path)} bytes")

    # Try to read just metadata
    try:
        metadata = pq.read_metadata(file_path)
        print("Metadata read successfully")
        print(f"Number of rows: {metadata.num_rows}")
        print(f"Number of columns: {metadata.num_columns}")
    except Exception as e:
        print(f"Error reading metadata: {e}")

    try:
      df= pd.read_parquet(
        file_path,
        engine='fastparquet'  # Instead of pyarrow
        )
    except Exception as e:
      print(f"Fastparquet error: {e}")    
    return df  

In [ ]:
# Phase 1.1: Load Data
# ====================

print("Loading dataset...")
all_reviews = read_parquet_with_checks("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/merged_700k_classified_with_ethnicity_14012026.parquet")
all_reviews.info()
print(f"Dataset shape: {all_reviews.shape}")
print(f"Columns: {all_reviews.columns.tolist()}")

In [ ]:
##chage state_nr to int dtype
all_reviews['state_nr'] = all_reviews['state_nr'].astype('Int64')

In [ ]:
#demographics and political  data
demo_pol_data = pd.read_csv("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/voters_demographic_data.csv")
demo_pol_data.info()
print(f"Dataset shape: {demo_pol_data.shape}")
print(f"Columns: {demo_pol_data.columns.tolist()}")

In [ ]:
#attemp merger
all_reviews = all_reviews.merge(demo_pol_data, on=['constituency_nr', 'constituency', 'state_nr', 'state'], how='left')
all_reviews.info()
print(f"Dataset shape after merge: {all_reviews.shape}")
print(f"Columns after merge: {all_reviews.columns.tolist()}")

In [ ]:
all_reviews.constituency.nunique()

## Phase 1.2: Define Racial Discourse Outcome


In [ ]:
# Phase 1.2: Define Racial Discourse Outcome
# ===========================================

# Check existing racial discourse columns
print("Racial discourse columns info:")
print(all_reviews['racial_connotations'].value_counts(dropna=False).head())
print(all_reviews['connotation_type'].value_counts(dropna=False).head())

# Create binary outcome variable
all_reviews['racial_discourse_binary'] = np.where(
    all_reviews['racial_connotations'].str.lower().isin(['true', 'yes', '1', 'y']), 1, 0
)

# Create categorical outcome (if needed)
def categorize_discourse(row):
    if row['racial_connotations'].lower() in ['true', 'yes', '1', 'y']:
        if 'pleasant' in str(row['connotation_type']).lower():
            return 'pleasant'
        elif 'discriminatory' in str(row['connotation_type']).lower():
            return 'discriminatory'
        else:
            return 'ambiguous'
    else:
        return 'none'

all_reviews['racial_discourse_category'] = all_reviews.apply(categorize_discourse, axis=1)

print("\nOutcome variable distribution:")
print(f"Binary (0/1): {all_reviews['racial_discourse_binary'].value_counts(normalize=True)}")
print(f"Categories: {all_reviews['racial_discourse_category'].value_counts(normalize=True)}")

## Phase 1.3: Aggregate to Constituency Level

In [ ]:
# Phase 1.3: Aggregate to Constituency Level
# ===========================================

# Check for missing constituency data
print(f"Missing constituency codes: {all_reviews['constituency_nr'].isna().sum()} ({all_reviews['constituency_nr'].isna().mean():.1%})")

# Remove rows without constituency for spatial analysis
all_reviews_spatial = all_reviews.dropna(subset=['constituency_nr']).copy()
print(f"Rows with constituency data: {all_reviews_spatial.shape[0]}")

# Aggregate to constituency level
constituency_stats = []

for const_code in all_reviews_spatial['constituency_nr'].unique():
    const_data = all_reviews_spatial[all_reviews_spatial['constituency_nr'] == const_code]
    
    # Calculate racial discourse metrics
    racial_rate = const_data['racial_discourse_binary'].mean()
    racial_count = const_data['racial_discourse_binary'].sum()
    total_reviews = len(const_data)
    pool_count = const_data['place_id'].nunique()
    
    # Get constituency characteristics (take first non-null value)
    constituency_row = {
        'constituency_nr': const_code,
        'constituency_name': const_data['constituency'].iloc[0] if 'constituency' in const_data.columns else None,
        'racial_discourse_rate': racial_rate,
        'racial_discourse_count': racial_count,
        'total_reviews': total_reviews,
        'unique_pools': pool_count,
        'latitude': const_data['latitude'].median(),
        'longitude': const_data['longitude'].median()
    }
    
    # Add demographic variables (if available)
    demo_cols = ['population_31.12.2023_foreign_nationals(%)', 
                 'population_density_31.12.2023 (inhabitants_per_km²)',
                 'total_population',
                 'age from ... to ... years on 31.12.2023 - under 18 (%)',
                 'age from ... to ... years on 31.12.2023 - 25-34 (%)']
    
    for col in demo_cols:
        if col in const_data.columns:
            constituency_row[col] = const_data[col].median()
    
    # Add political variables (vote shares)
    party_cols = ['afd', 'greens', 'spd', 'cdu', 'the left', 'fdp']
    for col in party_cols:
        if col in const_data.columns:
            constituency_row[f'vote_{col}'] = const_data[col].median()
    
    constituency_stats.append(constituency_row)

# Create constituency-level dataframe
const_all_reviews = pd.DataFrame(constituency_stats)
print(f"\nConstituency-level dataset shape: {const_all_reviews.shape}")
print(f"Number of constituencies: {const_all_reviews['constituency_nr'].nunique()}")

In [ ]:
const_all_reviews 

## Phase 1.4: Prepare Predictor Sets

In [ ]:
# Phase 1.4: Prepare Predictor Sets
# ==================================

# Define predictor sets
predictor_sets = {
    'demographic': [
        'population_31.12.2023_foreign_nationals(%)',
        'population_density_31.12.2023 (inhabitants_per_km²)',
        'total_population',
        'age from ... to ... years on 31.12.2023 - under 18 (%)',
        'age from ... to ... years on 31.12.2023 - 25-34 (%)',
        'age from ... to ... years on 31.12.2023 - 60-74 (%)'
    ],
    
    'political': [
        'vote_afd',
        'vote_greens',
        'vote_spd',
        'vote_cdu',
        'vote_the left',
        'vote_fdp'
    ],
    
    'structural': [
        'total_population',
        'population_density_31.12.2023 (inhabitants_per_km²)',
        'population increase (+) or decrease (-) in 2023 - birth balance (per 1000 inhabitants)',
        'population increase (+) or decrease (-) in 2022 - migration balance (per 1000 inhabitants)'
    ]
}

# Clean and prepare data
def prepare_constituency_data(const_df):
    """Clean and prepare constituency-level data for analysis."""
    
    # Drop rows with missing outcome
    const_df_clean = const_df.dropna(subset=['racial_discourse_rate']).copy()
    
    # Handle missing predictors - impute with median
    for predictor_set in predictor_sets.values():
        for predictor in predictor_set:
            if predictor in const_df_clean.columns:
                const_df_clean[predictor] = const_df_clean[predictor].fillna(
                    const_df_clean[predictor].median()
                )
    
    # Log transform skewed variables
    skewed_cols = ['racial_discourse_count', 'total_reviews', 'total_population']
    for col in skewed_cols:
        if col in const_df_clean.columns and const_df_clean[col].min() > 0:
            const_df_clean[f'log_{col}'] = np.log1p(const_df_clean[col])
    
    # Standardize predictors for comparison
    scaler = StandardScaler()
    all_predictors = []
    for predictors in predictor_sets.values():
        all_predictors.extend([p for p in predictors if p in const_df_clean.columns])
    
    for predictor in all_predictors:
        const_df_clean[f'z_{predictor}'] = scaler.fit_transform(
            const_df_clean[[predictor]]
        )
    
    return const_df_clean

const_df_clean = prepare_constituency_data(const_all_reviews)
print(f"Cleaned constituency dataset shape: {const_df_clean.shape}")
print(f"Available columns: {const_df_clean.columns.tolist()}")

In [ ]:
const_df_clean

## Phase 2.1: Correlation Analysis

In [ ]:
# Phase 2.1: Correlation Analysis
# ================================

# Select key variables for correlation analysis
corr_vars = [
    'racial_discourse_rate',
    'population_31.12.2023_foreign_nationals(%)',
    'vote_afd',
    'vote_greens',
    'population_density_31.12.2023 (inhabitants_per_km²)',
    'total_reviews'
]
#rename some columns for better visualization
const_df_clean = const_df_clean.rename(columns={
    'population_31.12.2023_foreign_nationals(%)': 'pct_foreign_nationals',
    'population_density_31.12.2023 (inhabitants_per_km²)': 'population_density'
})
# Update corr_vars to use the new column names
corr_vars = [
    'racial_discourse_rate',
    'pct_foreign_nationals',
    'vote_afd',
    'vote_greens',
    'population_density',
    'total_reviews'
]
# Filter to available variables
available_corr_vars = [v for v in corr_vars if v in const_df_clean.columns]
corr_matrix = const_df_clean[available_corr_vars].corr(method='spearman')

print("Spearman Correlation Matrix:")
print(corr_matrix.round(3))

# Statistical significance of correlations
print("\nCorrelation with racial_discourse_rate (with p-values):")
for var in available_corr_vars[1:]:  # Skip the first (racial_discourse_rate itself)
    if var in const_df_clean.columns:
        corr, p_value = stats.spearmanr(
            const_df_clean['racial_discourse_rate'].dropna(),
            const_df_clean[var].dropna()
        )
        print(f"{var:50} Correlation: {corr:.3f}, p-value: {p_value:.4f}")
        if p_value < 0.05:
            print(f"  → Statistically significant at p < 0.05")

# Visualize correlation matrix
plt.figure(figsize=(18, 16))

# Create a mask for the upper triangle (optional)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Create heatmap with larger fonts
heatmap = sns.heatmap(
    corr_matrix, 
    #mask=mask, 
    annot=True, 
    cmap='coolwarm', 
    center=0,
    square=True, 
    linewidths=1, 
    cbar_kws={"shrink": 0.8, "label": "Correlation Coefficient"},
    annot_kws={"size": 30, "weight": "bold"},  # Increased annotation font size
    fmt=".2f"  # Format annotations to 2 decimal places
)

# Increase font size of tick labels (variable names)
plt.xticks(fontsize=30, rotation=45, ha='right')
plt.yticks(fontsize=30, rotation=0)

# Increase font size of colorbar label and ticks
cbar = heatmap.collections[0].colorbar
cbar.ax.tick_params(labelsize=30)
cbar.ax.yaxis.label.set_size(30)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## Phase 2.2: Group Comparisons

In [ ]:
# Phase 2.2: Group Comparisons
# =============================

# Create quartile groups for key predictors
const_df_clean['foreign_pop_quartile'] = pd.qcut(
    const_df_clean['population_31.12.2023_foreign_nationals(%)'], 
    4, 
    labels=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)']
)

const_df_clean['afd_vote_quartile'] = pd.qcut(
    const_df_clean['vote_afd'], 
    4, 
    labels=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)']
)

# Compare racial discourse rates across quartiles
print("Racial discourse rate by foreign population quartile:")
quartile_stats = const_df_clean.groupby('foreign_pop_quartile').agg({
    'racial_discourse_rate': ['mean', 'std', 'count']
}).round(4)
print(quartile_stats)

print("\nRacial discourse rate by AfD vote share quartile:")
afd_stats = const_df_clean.groupby('afd_vote_quartile').agg({
    'racial_discourse_rate': ['mean', 'std', 'count']
}).round(4)
print(afd_stats)

# Statistical tests for group differences
print("\nANOVA Test for Foreign Population Quartiles:")
quartile_groups = [
    const_df_clean[const_df_clean['foreign_pop_quartile'] == q]['racial_discourse_rate']
    for q in const_df_clean['foreign_pop_quartile'].cat.categories
]
f_stat, p_value = stats.f_oneway(*quartile_groups)
print(f"F-statistic: {f_stat:.3f}, p-value: {p_value:.4f}")

# T-test for extreme quartiles
q1_data = const_df_clean[const_df_clean['foreign_pop_quartile'] == 'Q1 (Lowest)']['racial_discourse_rate']
q4_data = const_df_clean[const_df_clean['foreign_pop_quartile'] == 'Q4 (Highest)']['racial_discourse_rate']
t_stat, t_p = stats.ttest_ind(q1_data, q4_data, equal_var=False)
print(f"\nT-test Q1 vs Q4 (foreign population): t={t_stat:.3f}, p={t_p:.4f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Boxplot for foreign population quartiles
sns.boxplot(data=const_df_clean, x='foreign_pop_quartile', y='racial_discourse_rate',
            ax=axes[0], palette='viridis')
axes[0].set_title('Racial Discourse Rate by Foreign Population Quartile')
axes[0].set_xlabel('Foreign Population Quartile')
axes[0].set_ylabel('Racial Discourse Rate')
axes[0].tick_params(axis='x', rotation=45)

# Scatter plot with regression line
sns.regplot(data=const_df_clean, 
            x='population_31.12.2023_foreign_nationals(%)', 
            y='racial_discourse_rate',
            ax=axes[1], scatter_kws={'alpha': 0.6}, line_kws={'color': 'red'})
axes[1].set_title('Racial Discourse Rate vs Foreign Population %')
axes[1].set_xlabel('Foreign Population (%)')
axes[1].set_ylabel('Racial Discourse Rate')

plt.tight_layout()
plt.savefig('group_comparisons.png', dpi=300, bbox_inches='tight')
plt.show()

## Phase 2.3: Spatial Autocorrelation Analysis

In [ ]:
# Phase 2.3: Spatial Autocorrelation Analysis
# ===========================================

def check_spatial_autocorrelation(const_df_clean):
    """Check for spatial clustering of racial discourse."""
    
    # Create spatial weights matrix (Queen contiguity would be ideal but requires shapefile)
    # For now, use distance-based weights
    
    # Get coordinates
    coords = const_df_clean[['latitude', 'longitude']].values
    
    # Calculate distance matrix
    dist_matrix = squareform(pdist(coords, metric='euclidean'))
    
    # Create inverse distance weights
    threshold = np.percentile(dist_matrix[dist_matrix > 0], 10)  # 10th percentile as threshold
    weights_matrix = 1.0 / (dist_matrix + 0.001)  # Add small constant to avoid division by zero
    weights_matrix[dist_matrix > threshold] = 0  # Only consider nearby neighbors
    np.fill_diagonal(weights_matrix, 0)  # Zero diagonal
    
    # Calculate Moran's I
    racial_rates = const_df_clean['racial_discourse_rate'].values
    
    # Standardize racial rates
    racial_rates_std = (racial_rates - racial_rates.mean()) / racial_rates.std()
    
    # Manual calculation of Moran's I
    n = len(racial_rates_std)
    S0 = weights_matrix.sum()
    
    # Numerator: sum of weighted products
    numerator = 0
    for i in range(n):
        for j in range(n):
            numerator += weights_matrix[i, j] * racial_rates_std[i] * racial_rates_std[j]
    
    # Denominator: sum of squares
    denominator = (racial_rates_std ** 2).sum()
    
    morans_i = (n / S0) * (numerator / denominator)
    
    # Calculate expected value and variance for significance test
    expected_i = -1 / (n - 1)
    
    # Simplified variance calculation
    s1 = (weights_matrix + weights_matrix.T).sum() / 2
    s2 = ((weights_matrix.sum(axis=1) + weights_matrix.sum(axis=0)) ** 2).sum()
    s0 = S0
    
    var_i = (n * ((n**2 - 3*n + 3)*s1 - n*s2 + 3*s0**2) -
             (n * (n-1) * ((n**2 - n)*s1 - 2*n*s2 + 6*s0**2))) / \
            ((n-1)*(n-2)*(n-3)*s0**2) - expected_i**2
    
    # Z-score
    z_score = (morans_i - expected_i) / np.sqrt(var_i)
    p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))
    
    print(f"\nSpatial Autocorrelation Analysis:")
    print(f"Moran's I: {morans_i:.4f}")
    print(f"Expected I: {expected_i:.4f}")
    print(f"Variance: {var_i:.6f}")
    print(f"Z-score: {z_score:.4f}")
    print(f"P-value: {p_value:.4f}")
    
    if p_value < 0.05:
        if morans_i > expected_i:
            print("→ Statistically significant positive spatial autocorrelation")
            print("  (Racial discourse rates are spatially clustered)")
        else:
            print("→ Statistically significant negative spatial autocorrelation")
            print("  (Racial discourse rates are spatially dispersed)")
    else:
        print("→ No significant spatial autocorrelation detected")
    
    return morans_i, z_score, p_value

# Run spatial autocorrelation check
morans_i, z_score, p_value = check_spatial_autocorrelation(const_df_clean)

# Visualize spatial distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter plot of racial discourse rate
axes[0].scatter(const_df_clean['longitude'], const_df_clean['latitude'],
                c=const_df_clean['racial_discourse_rate'],
                cmap='viridis', s=50, alpha=0.7, edgecolor='k', linewidth=0.5)
axes[0].set_title('Geographic Distribution of Racial Discourse Rate')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
plt.colorbar(axes[0].collections[0], ax=axes[0], label='Racial Discourse Rate')

# Histogram with normal distribution overlay
axes[1].hist(const_df_clean['racial_discourse_rate'], bins=30, density=True,
             alpha=0.7, color='steelblue', edgecolor='black')
axes[1].axvline(const_df_clean['racial_discourse_rate'].mean(), color='red',
                linestyle='--', linewidth=2, label=f'Mean: {const_df_clean["racial_discourse_rate"].mean():.3f}')
axes[1].set_title('Distribution of Racial Discourse Rates')
axes[1].set_xlabel('Racial Discourse Rate')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.savefig('spatial_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm  # For normal distribution overlay if needed

# First plot: Scatter plot of racial discourse rate
plt.figure(figsize=(12, 8))
scatter = plt.scatter(const_df_clean['longitude'], const_df_clean['latitude'],
                      c=const_df_clean['racial_discourse_rate'],
                      cmap='viridis', s=50, alpha=0.7, edgecolor='k', linewidth=0.5)
plt.title('Geographic Distribution of Racial Discourse Rate')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.colorbar(scatter, label='Racial Discourse Rate')
plt.tight_layout()
plt.savefig('geographic_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# Second plot: Histogram with normal distribution overlay
plt.figure(figsize=(18, 16))

# Create histogram
n, bins, patches = plt.hist(const_df_clean['racial_discourse_rate'], bins=30, density=True,
                           alpha=0.7, color='steelblue', edgecolor='black')

# Add normal distribution overlay (optional enhancement)
mean = const_df_clean['racial_discourse_rate'].mean()
std = const_df_clean['racial_discourse_rate'].std()
x = np.linspace(const_df_clean['racial_discourse_rate'].min(), 
                const_df_clean['racial_discourse_rate'].max(), 100)
p = norm.pdf(x, mean, std)
plt.plot(x, p, 'k-', linewidth=2, label=f'Normal dist (μ={mean:.3f}, σ={std:.3f})')

# Add mean line
plt.axvline(mean, color='red', linestyle='--', linewidth=2, 
            label=f'Mean: {mean:.3f}')

#plt.title('Distribution of Racial Discourse Rates')
plt.xlabel('Racial Discourse Rate', fontsize=30)
plt.xticks(fontsize=30)
plt.ylabel('Density', fontsize=30)
plt.yticks(fontsize=30)
plt.legend(fontsize=30)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('rd_distribution_histogram.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Enhanced Spatial Visualization Code
# ===================================

# If you have constituency shapefiles:
import geopandas as gpd

def create_choropleth_map(const_df_clean, shapefile_path):
    """
    Create a proper choropleth map with constituency boundaries.
    
    Parameters:
    const_df_clean: Your constituency-level dataframe
    shapefile_path: Path to German constituency shapefile
    """
    # Load shapefile
    gdf = gpd.read_file(shapefile_path)
    
    # Merge with your data (assuming 'constituency_code' matches)
    gdf = gdf.merge(const_df_clean, 
                    left_on='WKR_NR',  # Adjust to match your shapefile's constituency code field
                    right_on='constituency_nr',
                    how='left')
    
    # Create choropleth
    fig, ax = plt.subplots(1, 1, figsize=(15, 10))
    
    # Plot with color gradient
    gdf.plot(column='racial_discourse_rate',
             cmap='RdYlBu_r',  # Red-Yellow-Blue reversed (red = high)
             linewidth=0.5,
             edgecolor='black',
             legend=True,
             legend_kwds={'label': "Racial Discourse Rate",
                          'orientation': "horizontal",
                          'shrink': 0.7},
             ax=ax,
             missing_kwds={'color': 'lightgrey', 
                          'label': 'Missing data',
                          'hatch': '///'})
    
    # Add context
    #ax.set_title('Spatial Distribution of Racial Discourse Rates\n(German Constituencies)', 
    #             fontsize=16, fontweight='bold')
    ax.set_axis_off()
    
    # Add annotations for key findings
    ax.text(0.02, 0.02, 
            f"Moran's I = {morans_i:.3f} (p < 0.001)\nSignificant spatial clustering",
            transform=ax.transAxes,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.savefig('constituency_choropleth.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return gdf

# Alternative: Create hexbin map if shapefile unavailable
def create_hexbin_map(const_df_clean):
    """
    Create density-based hexbin map when shapefiles aren't available.
    """
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    # Hexbin plot shows density of high-rate areas
    hb = ax.hexbin(const_df_clean['longitude'], 
                   const_df_clean['latitude'],
                   C=const_df_clean['racial_discourse_rate'],
                   gridsize=30,
                   cmap='RdYlBu_r',
                   reduce_C_function=np.mean)
    
    # Add colorbar
    cb = plt.colorbar(hb, ax=ax)
    cb.set_label('Mean Racial Discourse Rate')
    
    # Add context
    ax.set_title('Spatial Density of Racial Discourse\n(Hexbin Aggregation)', 
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    
    # Add Germany outline (simplified)
    # You could add a basemap here using contextily if needed
    
    plt.tight_layout()
    plt.savefig('hexbin_spatial_density.png', dpi=300, bbox_inches='tight')
    plt.show()

# Use one of these if possible
create_choropleth_map(const_df_clean, "/Users/i2scg/Documents/PHD/sm_project/btw25_geometrie_wahlkreise_shp_geo/btw25_geometrie_wahlkreise_shp_geo.shp")
create_hexbin_map(const_df_clean)

In [ ]:
def create_choropleth_map(const_df_clean, shapefile_path):
    """
    Create a proper choropleth map with constituency boundaries.
    
    Parameters:
    const_df_clean: Your constituency-level dataframe
    shapefile_path: Path to German constituency shapefile
    """
    # Load shapefile
    gdf = gpd.read_file(shapefile_path)
    
    # Merge with your data (assuming 'constituency_code' matches)
    gdf = gdf.merge(const_df_clean, 
                    left_on='WKR_NR',  # Adjust to match your shapefile's constituency code field
                    right_on='constituency_nr',
                    how='left')
    
    # Create choropleth
    fig, ax = plt.subplots(1, 1, figsize=(18, 16))
    
    # Plot with color gradient - REMOVE legend=True from here
    gdf.plot(column='racial_discourse_rate',
             cmap='RdYlBu_r',  # Red-Yellow-Blue reversed (red = high)
             linewidth=0.5,
             edgecolor='black',
             legend=False,  # Disable automatic legend
             ax=ax,
             missing_kwds={'color': 'lightgrey', 
                          'label': 'Missing data',
                          'hatch': '///'})
    
    # Add colorbar/legend manually on the side
    # Get the min and max values for the colorbar
    vmin = gdf['racial_discourse_rate'].min()
    vmax = gdf['racial_discourse_rate'].max()
    
    # Create a ScalarMappable for the colorbar
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    sm = plt.cm.ScalarMappable(cmap='RdYlBu_r', norm=norm)
    sm.set_array([])
    
    # Add colorbar on the right side
    cbar = plt.colorbar(sm, ax=ax, orientation='vertical', 
                        shrink=0.7, pad=0.02)
    cbar.set_label('Racial Discourse Rate', fontsize=30,labelpad=15)
    # Increase fontsize of colorbar tick labels
    cbar.ax.tick_params(labelsize=30)
    
    # Add context
    ax.set_axis_off()
    
    # Add annotations for key findings - MOVED to lower right corner
    # Changed coordinates from (0.02, 0.02) to (0.98, 0.02) and alignment to right
    ax.text(0.98, 0.02, 
            f"Moran's I = {morans_i:.3f} (p < 0.001)\nSignificant spatial clustering",
            transform=ax.transAxes, fontsize=30,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
            horizontalalignment='right',  # Align text to right
            verticalalignment='bottom')
    
    plt.tight_layout()
    plt.savefig('constituency_choropleth.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return gdf
create_choropleth_map(const_df_clean, "/Users/i2scg/Documents/PHD/sm_project/btw25_geometrie_wahlkreise_shp_geo/btw25_geometrie_wahlkreise_shp_geo.shp")

## Phase 3.1: Prepare Individual-Level Data for Multilevel Analysis

In [ ]:
# Phase 3.1: Prepare Individual-Level Data for Multilevel Analysis
# =================================================================

print("Preparing data for multilevel analysis...")

# Create individual-level dataset with constituency features
individual_cols = [
    'reviewer_rating', 'place_rating', 'total_reviews',
    'racial_discourse_binary', 'constituency_nr'
]

# Merge constituency-level features with individual data
if 'constituency_nr' in all_reviews.columns and 'constituency_nr' in const_df_clean.columns:
    # Create constituency features dictionary
    constituency_features = const_df_clean.set_index('constituency_nr')[
        ['racial_discourse_rate', 'population_31.12.2023_foreign_nationals(%)',
         'vote_afd', 'vote_greens', 'population_density_31.12.2023 (inhabitants_per_km²)']
    ].to_dict('index')
    
    # Add constituency features to individual data
    df_multilevel = all_reviews[individual_cols].copy()
    
    for feature in ['racial_discourse_rate', 'population_31.12.2023_foreign_nationals(%)',
                    'vote_afd', 'vote_greens', 'population_density_31.12.2023 (inhabitants_per_km²)']:
        df_multilevel[f'const_{feature}'] = df_multilevel['constituency_nr'].map(
            lambda x: constituency_features.get(x, {}).get(feature, np.nan)
        )
    
    # Remove rows with missing data
    df_multilevel_clean = df_multilevel.dropna()
    print(f"Multilevel dataset shape: {df_multilevel_clean.shape}")
    print(f"Number of constituencies in multilevel data: {df_multilevel_clean['constituency_nr'].nunique()}")
else:
    print("Constituency nr not found in both datasets")
    df_multilevel_clean = pd.DataFrame()

In [ ]:
df_multilevel_clean

## Phase 3.2: Multilevel Model Specification

In [ ]:
# Phase 3.2: Multilevel Model Specification
# ==========================================

if len(df_multilevel_clean) > 0:
    # Simple logistic regression (baseline)
    print("\n=== Baseline Logistic Regression (No Random Effects) ===")
    
    X_baseline = df_multilevel_clean[['reviewer_rating', 'place_rating', 
                                       'const_population_31.12.2023_foreign_nationals(%)',
                                       'const_vote_afd']]
    X_baseline = sm.add_constant(X_baseline)
    y_baseline = df_multilevel_clean['racial_discourse_binary']
    
    logit_model = sm.Logit(y_baseline, X_baseline)
    logit_result = logit_model.fit(disp=False)
    print(logit_result.summary())
    
    # Calculate pseudo R-squared
    llf = logit_result.llf
    llnull = logit_result.llnull
    pseudo_r2 = 1 - (llf / llnull)
    print(f"\nPseudo R-squared: {pseudo_r2:.4f}")
    
    # For multilevel model, we'll use a simpler approach with sklearn
    # (Full multilevel logistic regression requires specialized packages like statsmodels MixedLM)
    
    print("\n=== Grouped Analysis by Constituency ===")
    
    # Calculate ICC (Intraclass Correlation Coefficient)
    def calculate_icc(df, group_col, outcome_col):
        """Calculate Intraclass Correlation Coefficient."""
        # ANOVA approach
        groups = df.groupby(group_col)[outcome_col]
        n_groups = len(groups)
        n_obs = len(df)
        
        # Calculate between-group variance
        group_means = groups.mean()
        overall_mean = df[outcome_col].mean()
        ss_between = ((group_means - overall_mean) ** 2).sum() * groups.count().mean()
        
        # Calculate within-group variance
        ss_within = 0
        for name, group in groups:
            ss_within += ((group - group_means[name]) ** 2).sum()
        
        # Calculate ICC
        ms_between = ss_between / (n_groups - 1)
        ms_within = ss_within / (n_obs - n_groups)
        
        icc = (ms_between - ms_within) / (ms_between + (groups.count().mean() - 1) * ms_within)
        
        return icc, ms_between, ms_within
    
    icc, ms_between, ms_within = calculate_icc(
        df_multilevel_clean, 'constituency_nr', 'racial_discourse_binary'
    )
    
    print(f"Intraclass Correlation Coefficient (ICC): {icc:.4f}")
    print(f"Between-group variance: {ms_between:.4f}")
    print(f"Within-group variance: {ms_within:.4f}")
    print(f"→ Proportion of variance at constituency level: {icc:.1%}")
    
    if icc > 0.1:
        print("→ Substantial clustering at constituency level (ICC > 0.1)")
        print("→ Multilevel modeling is appropriate")
    else:
        print("→ Minimal clustering at constituency level")
        print("→ Simple regression may be sufficient")
else:
    print("Insufficient data for multilevel analysis")

## Phase 3.3: Constituency-Level Regression Analysis

In [ ]:
# Phase 3.3: Constituency-Level Regression Analysis
# ==================================================

print("\n=== Constituency-Level Regression Models ===")

# Prepare data
const_reg = const_df_clean.copy()

# Define outcome and predictors
outcome = 'racial_discourse_rate'

# Model 1: Demographic only
demo_predictors = [p for p in predictor_sets['demographic'] if p in const_reg.columns]

# Model 2: Political only
pol_predictors = [p for p in predictor_sets['political'] if p in const_reg.columns]

# Model 3: Combined
combined_predictors = demo_predictors + pol_predictors

# Function to run OLS regression
def run_ols_regression(df, outcome, predictors, model_name):
    """Run OLS regression and return results."""
    print(f"\n--- {model_name} ---")
    print(f"Predictors: {predictors}")
    
    # Prepare data
    X = df[predictors].copy()
    X = sm.add_constant(X)  # Add intercept
    y = df[outcome]
    
    # Remove rows with missing values
    valid_idx = X.notna().all(axis=1) & y.notna()
    X_valid = X[valid_idx]
    y_valid = y[valid_idx]
    
    if len(X_valid) < len(predictors) + 2:  # Need at least n+2 observations
        print("Insufficient data for this model")
        return None
    
    # Run regression
    model = sm.OLS(y_valid, X_valid)
    results = model.fit()
    
    # Print summary
    print(f"R-squared: {results.rsquared:.4f}")
    print(f"Adjusted R-squared: {results.rsquared_adj:.4f}")
    print(f"F-statistic: {results.fvalue:.2f} (p={results.f_pvalue:.4f})")
    print(f"Number of observations: {results.nobs}")
    
    # Print significant coefficients
    print("\nSignificant predictors (p < 0.05):")
    sig_coeffs = results.pvalues[results.pvalues < 0.05].sort_values()
    for predictor in sig_coeffs.index:
        if predictor != 'const':
            coeff = results.params[predictor]
            p_val = results.pvalues[predictor]
            print(f"  {predictor:50} β = {coeff:.4f}, p = {p_val:.4f}")
    
    # Check multicollinearity
    if len(predictors) > 1:
        print("\nVariance Inflation Factors (VIF):")
        vif_data = pd.DataFrame()
        vif_data["feature"] = X_valid.columns
        vif_data["VIF"] = [variance_inflation_factor(X_valid.values, i) 
                          for i in range(len(X_valid.columns))]
        for _, row in vif_data.iterrows():
            vif_val = row['VIF']
            if vif_val > 10:
                print(f"  {row['feature']:50} VIF = {vif_val:.2f} ← High multicollinearity!")
            elif vif_val > 5:
                print(f"  {row['feature']:50} VIF = {vif_val:.2f} ← Moderate multicollinearity")
            else:
                print(f"  {row['feature']:50} VIF = {vif_val:.2f}")
    
    return results

# Run models
model1 = run_ols_regression(const_reg, outcome, demo_predictors[:4], "Model 1: Demographic Factors")
model2 = run_ols_regression(const_reg, outcome, pol_predictors[:4], "Model 2: Political Factors")
model3 = run_ols_regression(const_reg, outcome, combined_predictors[:6], "Model 3: Combined Factors")

## Phase 4.1: Predictive Performance Assessment

In [ ]:
# Phase 4.1: Predictive Performance Assessment
# ============================================

print("\n=== Predictive Performance Assessment ===")

# Prepare data for prediction
predictor_cols = [
    'population_31.12.2023_foreign_nationals(%)',
    'vote_afd',
    'vote_greens',
    'population_density_31.12.2023 (inhabitants_per_km²)',
    'total_reviews',
    'age from ... to ... years on 31.12.2023 - 25-34 (%)'
]

# Filter to available columns
available_predictors = [p for p in predictor_cols if p in const_df_clean.columns]
X = const_df_clean[available_predictors].fillna(const_df_clean[available_predictors].median())
y = const_df_clean['racial_discourse_rate']

print(f"Using predictors: {available_predictors}")
print(f"Sample size: {len(X)}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Training set: {len(X_train)} observations")
print(f"Test set: {len(X_test)} observations")

# Train multiple models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    print(f"\n--- {name} ---")
    
    # Train model
    model.fit(X_train, y_train)
    
    # Predictions
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Calculate metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    
    print(f"Train RMSE: {train_rmse:.4f}")
    print(f"Test RMSE: {test_rmse:.4f}")
    print(f"Train R²: {train_r2:.4f}")
    print(f"Test R²: {test_r2:.4f}")
    
    # Cross-validation
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    print(f"Cross-validation R²: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")
    
    # Store results
    results[name] = {
        'model': model,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std()
    }

# Compare models
print("\n=== Model Comparison ===")
comparison_df = pd.DataFrame({
    name: [res['test_r2'], res['test_rmse'], res['cv_mean'], res['cv_std']]
    for name, res in results.items()
}, index=['Test R²', 'Test RMSE', 'CV R² Mean', 'CV R² Std']).T

print(comparison_df.round(4))

# Select best model
best_model_name = comparison_df['Test R²'].idxmax()
best_model = results[best_model_name]['model']
print(f"\nBest model: {best_model_name}")
print(f"Test R²: {comparison_df.loc[best_model_name, 'Test R²']:.4f}")

## Phase 4.2: Feature Importance Analysis

In [ ]:
# Phase 4.2: Feature Importance Analysis
# =======================================

print("\n=== Feature Importance Analysis ===")

# Get feature importance from best tree-based model
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feature_importance_df = pd.DataFrame({
        'feature': available_predictors,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    print("Feature Importance (from best tree-based model):")
    for _, row in feature_importance_df.iterrows():
        print(f"  {row['feature']:50} Importance: {row['importance']:.4f}")
    
    # Plot feature importance
    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance_df['feature'], feature_importance_df['importance'])
    plt.xlabel('Feature Importance')
    plt.title(f'Feature Importance - {best_model_name}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()

# Permutation importance (model-agnostic)
print("\nPermutation Importance:")
perm_importance = permutation_importance(
    best_model, X_test, y_test, n_repeats=10, random_state=42
)

perm_importance_df = pd.DataFrame({
    'feature': available_predictors,
    'importance_mean': perm_importance.importances_mean,
    'importance_std': perm_importance.importances_std
}).sort_values('importance_mean', ascending=False)

for _, row in perm_importance_df.iterrows():
    print(f"  {row['feature']:50} Importance: {row['importance_mean']:.4f} (±{row['importance_std']:.4f})")

## Phase 4.3: Intervention Site Identification

In [ ]:
# Phase 4.3: Intervention Site Identification
# ===========================================

print("\n=== Intervention Site Identification ===")

# Calculate residuals (prediction errors)
const_df_clean['predicted_rate'] = best_model.predict(X)
const_df_clean['residual'] = const_df_clean['racial_discourse_rate'] - const_df_clean['predicted_rate']
const_df_clean['abs_residual'] = np.abs(const_df_clean['residual'])

# Identify constituencies with largest prediction errors
print("\nConstituencies with largest over-prediction (Actual < Predicted):")
underpredicted = const_df_clean.nsmallest(10, 'residual')[
    ['constituency_name', 'racial_discourse_rate', 'predicted_rate', 'residual']
]
print(underpredicted.round(4))

print("\nConstituencies with largest under-prediction (Actual > Predicted):")
overpredicted = const_df_clean.nlargest(10, 'residual')[
    ['constituency_name', 'racial_discourse_rate', 'predicted_rate', 'residual']
]
print(overpredicted.round(4))

# Identify high-risk constituencies
# Define high-risk as: high predicted rate AND high foreign population
const_df_clean['high_risk_score'] = (
    const_df_clean['predicted_rate'].rank(pct=True) * 0.5 +
    const_df_clean['population_31.12.2023_foreign_nationals(%)'].rank(pct=True) * 0.3 +
    const_df_clean['vote_afd'].rank(pct=True) * 0.2
)

print("\nTop 10 High-Risk Constituencies for Intervention:")
high_risk = const_df_clean.nlargest(10, 'high_risk_score')[
    ['constituency_name', 'racial_discourse_rate', 'predicted_rate',
     'population_31.12.2023_foreign_nationals(%)', 'vote_afd', 'high_risk_score']
]
print(high_risk.round(4))

# Visualize prediction vs actual
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter plot: Actual vs Predicted
axes[0].scatter(const_df_clean['predicted_rate'], const_df_clean['racial_discourse_rate'],
                alpha=0.6, edgecolor='k', linewidth=0.5)
axes[0].plot([0, const_df_clean['racial_discourse_rate'].max()],
             [0, const_df_clean['racial_discourse_rate'].max()],
             'r--', label='Perfect prediction')
axes[0].set_xlabel('Predicted Racial Discourse Rate')
axes[0].set_ylabel('Actual Racial Discourse Rate')
axes[0].set_title('Predicted vs Actual Rates')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residuals plot
axes[1].scatter(const_df_clean['predicted_rate'], const_df_clean['residual'],
                alpha=0.6, edgecolor='k', linewidth=0.5)
axes[1].axhline(y=0, color='r', linestyle='--')
axes[1].set_xlabel('Predicted Racial Discourse Rate')
axes[1].set_ylabel('Residual (Actual - Predicted)')
axes[1].set_title('Residuals vs Predicted Values')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('prediction_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# First plot: Actual vs Predicted scatter plot
plt.figure(figsize=(18, 16))
plt.scatter(const_df_clean['predicted_rate'], const_df_clean['racial_discourse_rate'],
             s=150,  # Increase dot size here
            alpha=0.6, edgecolor='k', linewidth=0.5)
plt.plot([0, const_df_clean['racial_discourse_rate'].max()],
         [0, const_df_clean['racial_discourse_rate'].max()],
         'r--', label='Perfect prediction')
plt.xlabel('Predicted Racial Discourse Rate', fontsize=30)
plt.xticks(fontsize=30,rotation=45)
plt.yticks(fontsize=30)
plt.ylabel('Actual Racial Discourse Rate', fontsize=30)
#plt.title('Predicted vs Actual Rates', fontsize=30)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('predicted_vs_actual.png', dpi=300, bbox_inches='tight')
plt.show()

# Second plot: Residuals plot
plt.figure(figsize=(18, 16))
plt.scatter(const_df_clean['predicted_rate'], const_df_clean['residual'],
             s=150,  # Increase dot size here
            alpha=0.6, edgecolor='k', linewidth=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Racial Discourse Rate', fontsize=30)
plt.xticks(fontsize=30,rotation=45)
plt.ylabel('Residual (Actual - Predicted)', fontsize=30)
plt.yticks(fontsize=30)
#plt.title('Residuals vs Predicted Values', fontsize=30)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('residuals_plot.png', dpi=300, bbox_inches='tight')
plt.show()

## Phase 5: Generate Summary Report

In [ ]:
# Phase 5: Generate Summary Report
# =================================

print("\n" + "="*80)
print("ANALYSIS SUMMARY REPORT")
print("="*80)

print("\n1. DATA OVERVIEW")
print("-" * 40)
print(f"Total reviews analyzed: {len(all_reviews):,}")
print(f"Constituencies with data: {const_df_clean['constituency_nr'].nunique()}")
print(f"Average racial discourse rate: {const_df_clean['racial_discourse_rate'].mean():.3%}")
print(f"Range: {const_df_clean['racial_discourse_rate'].min():.3%} to {const_df_clean['racial_discourse_rate'].max():.3%}")

print("\n2. KEY CORRELATIONS WITH RACIAL DISCOURSE")
print("-" * 40)
print(f"Foreign population %: r = {corr_matrix.loc['racial_discourse_rate', 'population_31.12.2023_foreign_nationals(%)']:.3f}")
print(f"AfD vote share: r = {corr_matrix.loc['racial_discourse_rate', 'vote_afd']:.3f}")
print(f"Population density: r = {corr_matrix.loc['racial_discourse_rate', 'population_density_31.12.2023 (inhabitants_per_km²)']:.3f}")

print("\n3. MULTILEVEL ANALYSIS FINDINGS")
print("-" * 40)
print(f"Intraclass Correlation (ICC): {icc:.4f}")
print(f"→ {icc:.1%} of variance at constituency level")
if icc > 0.1:
    print("→ Significant clustering: Multilevel modeling appropriate")

print("\n4. PREDICTIVE MODEL PERFORMANCE")
print("-" * 40)
print(f"Best model: {best_model_name}")
print(f"Test R²: {results[best_model_name]['test_r2']:.4f}")
print(f"Cross-validation R²: {results[best_model_name]['cv_mean']:.4f} (±{results[best_model_name]['cv_std']:.4f})")

print("\n5. MOST IMPORTANT PREDICTORS")
print("-" * 40)
if 'feature_importance_df' in locals():
    top_features = feature_importance_df.head(5)
    for i, (_, row) in enumerate(top_features.iterrows(), 1):
        print(f"{i}. {row['feature']}: Importance = {row['importance']:.4f}")

print("\n6. INTERVENTION RECOMMENDATIONS")
print("-" * 40)
print(f"High-risk constituencies identified: {len(high_risk)}")
print("Primary risk factors identified:")
print("  1. High predicted racial discourse rates")
print("  2. High foreign population percentages")
print("  3. High AfD vote shares")

print("\n7. SPATIAL PATTERNS")
print("-" * 40)
print(f"Moran's I: {morans_i:.4f} (p = {p_value:.4f})")
if p_value < 0.05:
    if morans_i > 0:
        print("→ Significant spatial clustering detected")
        print("→ Racial discourse rates are geographically concentrated")
    else:
        print("→ Significant spatial dispersion detected")

print("\n8. KEY TAKEAWAYS FOR RQ2")
print("-" * 40)
print("✓ Racial discourse can be predicted from structural factors")
print(f"✓ Model explains ~{results[best_model_name]['test_r2']*100:.1f}% of spatial variation")
print("✓ Key predictors: Foreign population %, AfD vote share, population density")
print(f"✓ {len(high_risk)} constituencies identified as high-priority intervention sites")
print("✓ Spatial clustering suggests neighborhood-level contagion effects")

print("\n" + "="*80)
print("Analysis complete. Results saved to:")
print("1. correlation_matrix.png")
print("2. group_comparisons.png")
print("3. spatial_distribution.png")
print("4. feature_importance.png")
print("5. prediction_analysis.png")
print("="*80)

# Save results to CSV for further analysis
const_df_clean.to_csv('constituency_analysis_results.csv', index=False)
print("\nDetailed results saved to 'constituency_analysis_results.csv'")